# F2 — The ML Landscape

**Fonte:** https://insper.github.io/ML/2026.2/classes/ml-landscape/ (PT: `.../ML/pt/2026.2/classes/ml-landscape/`)

> **Nota sobre o formato deste notebook.** A própria página diz: *"Both labs run from this page:
> there is no notebook to download. Open a blank Colab notebook ... paste each code block into its
> own cell and run them in order."* Este notebook reproduz exatamente essa instrução (regra 3.3 do
> AGENTS.md): cada bloco de código abaixo é colado **verbatim** da página, na ordem exata (passos 1
> a 7), seguido da minha resposta ao Predict/Simulação correspondente. Os outputs impressos abaixo
> foram gerados executando este notebook do zero — todos batem com os publicados na página, porque
> o código usa `seed`/`random_state` fixos.

## 1. Learning paradigms

- **Supervisionado:** dataset com entradas $x$ e saídas desejadas $y$ (rótulos); objetivo: aprender
  $f$ tal que $f(x)\approx y$ em dados novos. *Regressão* ($y$ contínuo) ou *classificação* ($y$
  categórico).
- **Não supervisionado:** só $x$, sem rótulos; objetivo: descobrir estrutura (clustering, redução de
  dimensionalidade, topic modeling).
- **Por reforço:** um agente interage com um ambiente, recebe recompensas e aprende uma política;
  fora do escopo deste curso, mas bom reconhecer no mapa.
- **No meio do caminho:** aprendizado semi-supervisionado, auto-supervisionado e supervisão fraca.

## 2. The ML workflow

Um modelo é uma pequena parte de um processo iterativo maior: **enquadrar o problema → obter dados →
preparar → treinar → avaliar → implantar**, e o laço nunca termina (modelos implantados decaem —
*drift* — monitorar e retreinar faz parte do trabalho). A maior parte do esforço real está nos passos
2–3 (dados), não no ajuste do modelo.

## 3. Generalization: the central problem

**Underfitting:** o modelo é simples demais para capturar o padrão — desempenho ruim mesmo no treino.
**Overfitting:** o modelo captura ruído como se fosse sinal — excelente no treino, ruim em dados
novos. **A regra de ouro:** nunca avaliar um modelo nos dados em que ele foi treinado; o conjunto de
teste deve simular o futuro — nunca visto, nunca tocado, usado uma vez.

## Fundamental lab — the shape of the error

### Passo 1 — um mundo que você controla

**Predict.** Os pontos são a curva verdadeira mais ruído com desvio-padrão 0,3. Um modelo perfeito,
que encontrasse a curva exatamente, conseguiria zerar o erro nos pontos novos? Por quê?

**Resposta (antes de rodar):** Não. O ruído faz parte de cada ponto novo, e nenhum modelo pode
prevê-lo — mesmo a curva verdadeira erraria cada ponto novo pelo próprio ruído. O erro quadrático
médio da curva verdadeira seria a variância do ruído, $0{,}3^2=0{,}09$: esse é o piso, não zero.

In [1]:
import numpy as np

def true_curve(x):
    return 0.9 * np.sin(2.2 * np.pi * x) + 0.6 * x

def draw(n, seed):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = true_curve(x) + rng.normal(0, 0.3, n)        # the curve plus noise
    return x, y

x_train, y_train = draw(15, seed=0)                  # what the model learns from
x_new, y_new = draw(1000, seed=1)                    # the future: never shown to the model
print(len(x_train), 'training points ·', len(x_new), 'new points')
print('first three training points:', ', '.join(f'({a:.2f}, {b:.2f})' for a, b in zip(x_train[:3], y_train[:3])))
print('noise variance, the floor of the error on new points:', 0.3 ** 2)

15 training points · 1000 new points
first three training points: (0.64, -0.69), (0.27, 0.86), (0.04, 0.18)
noise variance, the floor of the error on new points: 0.09


**Resposta (depois de rodar):** Confirmado — o output mostra exatamente o piso de $0{,}09$ que
previ. Quinze pontos para aprender, mil para julgar; em um projeto real ninguém nos dá a função
`draw` — aqui ela deixa ver o erro no futuro diretamente.

### Passo 2 — três modelos, a olho

**Predict.** Polinômios de grau 1, 3 e 12 são ajustados aos mesmos 15 pontos. Ordene-os pelo erro de
treino. A ordem pelo erro em pontos novos será a mesma?

**Resposta (antes de rodar):** Pelo erro de treino, quanto mais flexível (maior grau), menor o erro:
ordem 12, 3, 1. Não espero que a ordem se mantenha para pontos novos — um polinômio de grau 12 com
só 15 pontos deve estar memorizando ruído, não a curva.

In [2]:
from numpy.polynomial import Polynomial

def mse(model, x, y):
    return np.mean((y - model(x)) ** 2)              # mean squared error

for degree in [1, 3, 12]:
    model = Polynomial.fit(x_train, y_train, degree)
    print(f"degree {degree:2d}: train {mse(model, x_train, y_train):.3f} · new {mse(model, x_new, y_new):.3f}")

degree  1: train 0.208 · new 0.435
degree  3: train 0.036 · new 0.119
degree 12: train 0.004 · new 92.010


**Resposta (depois de rodar):** Confirmado. Pelo erro de treino a ordem é 12, 3, 1 (0,004 <
0,036 < 0,208): quanto mais flexível, melhor. Pelo erro em pontos novos a ordem se inverte para o
grau 12: fica 3, 1, 12 (0,119 < 0,435 < 92,010) — o grau 12 não é um pouco pior, é catastroficamente
pior. As duas ordens discordam, e só a segunda descreve o que o modelo fará depois de implantado.

### Passo 3 — a curva completa

**Predict.** Agora todo grau de 0 a 14. Antes de rodar, esboce os dois erros contra o grau. Qual só
cai? Onde o outro tem seu mínimo?

**Resposta (antes de rodar):** O erro de treino só deve cair (mais flexibilidade sempre ajuda a
encaixar os 15 pontos vistos). O erro em pontos novos deve ter formato de U: alto nos graus baixos
(underfitting), baixo numa faixa intermediária, e explodindo nos graus altos (overfitting).

In [3]:
print('degree   train            new')
for degree in range(15):
    model = Polynomial.fit(x_train, y_train, degree)
    print(f"{degree:6d} {mse(model, x_train, y_train):7.3f} {mse(model, x_new, y_new):14,.3f}")

degree   train            new
     0   0.208          0.434
     1   0.208          0.435
     2   0.173          0.514
     3   0.036          0.119
     4   0.036          0.118
     5   0.035          0.111
     6   0.035          0.107
     7   0.028          0.514
     8   0.028          0.513
     9   0.023          1.765
    10   0.007         25.078
    11   0.006         83.926
    12   0.004         92.010
    13   0.002     79,229.911
    14   0.000 455,507,180.697


**Resposta (depois de rodar):** Confirmado — a coluna de treino só cai, chegando a 0,000 no
grau 14 (15 coeficientes para 15 pontos: a curva passa exatamente por todos eles). A coluna de dados
novos tem o U esperado: graus 0–2 fazem underfitting (0,43 a 0,51, os dois erros altos); graus 3–6
ficam entre 0,107 e 0,119, perto do piso de 0,09; do grau 7 em diante o overfitting assume o controle,
chegando a 455 milhões no grau 14. O grau 0 (reta horizontal na média do treino) é o modelo mais
simples de todos — a referência (*baseline*) que qualquer modelo precisa superar.

**Simulação: mais dados.** Com 100 pontos de treino em vez de 15, o polinômio de grau 14 ainda
explode em pontos novos?

**Resposta (antes de rodar):** Não deveria explodir mais: com 100 pontos, 15 coeficientes não
conseguem mais passar exatamente por todos os pontos, então a curva é forçada a seguir a tendência em
vez de memorizar ruído específico.

In [4]:
x_big, y_big = draw(100, seed=0)
for degree in [1, 3, 5, 8, 12, 14]:
    model = Polynomial.fit(x_big, y_big, degree)
    print(f"degree {degree:2d}, 100 points: train {mse(model, x_big, y_big):.3f} · new {mse(model, x_new, y_new):.3f}")

degree  1, 100 points: train 0.392 · new 0.327
degree  3, 100 points: train 0.092 · new 0.114
degree  5, 100 points: train 0.079 · new 0.104
degree  8, 100 points: train 0.076 · new 0.109
degree 12, 100 points: train 0.075 · new 0.111
degree 14, 100 points: train 0.074 · new 0.114


**Resposta (depois de rodar):** Confirmado — com 100 pontos o grau 14 marca 0,114 em pontos
novos, igual ao grau 3, em vez de 455 milhões. "Flexível demais" é sempre relativo à quantidade de
dados disponível: os mesmos 15 coeficientes que memorizavam 15 pontos não conseguem mais memorizar
100.

### Passo 4 — um modelo que memoriza

**Predict.** O "modelo" mais simples possível guarda os pontos de treino e responde, para qualquer
$x$, o $y$ do ponto de treino mais próximo. Qual será seu erro de treino? Em pontos novos, ele vence
o polinômio de grau 3?

**Resposta (antes de rodar):** No treino, erro zero por construção (o ponto de treino mais próximo de
si mesmo é ele mesmo). Em pontos novos, não espero que vença o grau 3: cada resposta carrega o ruído
específico do único ponto copiado, em vez de uma média que cancela parte do ruído.

In [5]:
def memorizer(x_seen, y_seen):
    def model(x):
        nearest = np.abs(x[:, None] - x_seen[None, :]).argmin(axis=1)    # index of the closest seen x
        return y_seen[nearest]
    return model

memo = memorizer(x_train, y_train)
cubic = Polynomial.fit(x_train, y_train, 3)
print(f"memorizer: train {mse(memo, x_train, y_train):.3f} · new {mse(memo, x_new, y_new):.3f}")
print(f"degree 3:  train {mse(cubic, x_train, y_train):.3f} · new {mse(cubic, x_new, y_new):.3f}")

memorizer: train 0.000 · new 0.175
degree 3:  train 0.036 · new 0.119


**Resposta (depois de rodar):** Confirmado em ambas as previsões: erro de treino 0,000 para o
memorizador, e ele perde para o grau 3 em pontos novos (0,175 contra 0,119). Um erro de treino
perfeito é exatamente a cara de quem memorizou — e é por isso que a regra de ouro existe: julgando
pelos dados de treino, o memorizador pareceria melhor que o grau 3, quando é o contrário.

**Simulação: nada para aprender.** Embaralhando os rótulos de treino (sem padrão algum restante) e
construindo um memorizador sobre eles — qual o erro de treino agora? E comparado aos 0,434 do grau 0
(responder sempre a média)?

**Resposta (antes de rodar):** O erro de treino deve continuar 0,000 (o memorizador sempre acerta seu
próprio ponto, com ou sem padrão real). Em pontos novos deve ficar pior que a média simples (0,434),
porque não há mais nenhuma estrutura real para copiar, só ruído embaralhado.

In [6]:
y_random = np.random.default_rng(0).permutation(y_train)      # same y values, shuffled: no pattern left
memo_random = memorizer(x_train, y_random)
print(f"memorizer, shuffled y: train {mse(memo_random, x_train, y_random):.3f} · new {mse(memo_random, x_new, y_new):.3f}")

memorizer, shuffled y: train 0.000 · new 0.717


**Resposta (depois de rodar):** Confirmado — erro de treino continua 0,000, e em pontos novos
fica em 0,717, bem pior que os 0,434 de simplesmente responder a média. O erro de treino de um modelo
flexível o bastante para memorizar não diz absolutamente nada; só dados novos distinguem os dois
casos.

**Para encerrar a parte fundamental:**

1. O erro de treino diz se o modelo consegue se ajustar aos pontos que viu; ele **não** pode dizer
   como o modelo se comportará em dados que nunca viu — um modelo flexível sempre consegue um erro de
   treino baixo, memorizando em vez de aprender.
2. Cem pontos domaram o polinômio de grau 14 porque 15 coeficientes não bastam mais para passar
   exatamente por 100 pontos: a curva é forçada a seguir a tendência geral, e o ruído de 100 pontos se
   cancela mais do que o de 15.
3. Esta aula é um exemplo de aprendizado **supervisionado** com tarefa de **regressão** ($y$ é
   contínuo — a saída da curva mais ruído).

### Minha vez (fundamental)

Com `draw(30, seed=2)` dividido 50/50 em treino/teste (em vez de 1000 pontos "do futuro"), as duas
colunas mantêm o formato geral (treino só cai, teste em U), mas a coluna de teste fica muito mais
ruidosa: com só 15 pontos de teste em vez de 1000, cada erro médio é a média de poucas amostras, então
tem muito mais variância de uma repetição para outra — o mesmo problema de tamanho de amostra que
aparece em toda estimativa estatística.

**Paradigma e tarefa de cada caso:**

- (a) Demanda de eletricidade de amanhã, a partir de 5 anos de demanda horária: **supervisionado,
  regressão** ($y$ contínuo).
- (b) Agrupar músicas por atributos de áudio, sem gêneros dados: **não supervisionado, clustering**.
- (c) Marcar transações fraudulentas, a partir de transações passadas já marcadas por analistas:
  **supervisionado, classificação** ($y$ categórico: fraude/não fraude).
- (d) Robô aspirador que aprende uma rota a partir de uma pontuação recebida ao final de cada
  execução: **aprendizado por reforço** (recompensa ao final, não rótulo por exemplo).

**Um mundo mais silencioso** (ruído 0,1 em vez de 0,3): o piso do erro em pontos novos cai para
$0{,}1^2=0{,}01$, bem abaixo do piso anterior de 0,09, e o overfitting deve começar num grau **mais
alto** que antes — com menos ruído para memorizar, o modelo pode ser mais flexível antes que o
overfitting vença.

In [7]:
x_train_q, y_train_q = draw(15, seed=0)
rng_q = np.random.default_rng(0)
x_q = rng_q.uniform(0, 1, 15)
y_q = true_curve(x_q) + rng_q.normal(0, 0.1, 15)       # mesmo gerador, ruído 0.1
x_new_q = rng_q.uniform(0, 1, 1000)
y_new_q = true_curve(x_new_q) + rng_q.normal(0, 0.1, 1000)

print('degree   train            new  (ruído 0.1)')
for degree in range(15):
    model = Polynomial.fit(x_q, y_q, degree)
    print(f"{degree:6d} {mse(model, x_q, y_q):7.3f} {mse(model, x_new_q, y_new_q):14,.3f}")

degree   train            new  (ruído 0.1)
     0   0.183          0.339
     1   0.182          0.327
     2   0.144          0.386
     3   0.008          0.027
     4   0.006          0.020
     5   0.004          0.011
     6   0.004          0.011
     7   0.003          0.056
     8   0.003          0.055
     9   0.003          0.187
    10   0.001          2.720
    11   0.001          9.115
    12   0.000          9.958
    13   0.000      8,495.638
    14   0.000 49,530,816.655


**Confirmado:** com ruído 0,1 o piso é muito mais baixo (perto de 0,01) e a coluna de teste
permanece próxima do piso por mais graus antes de explodir, confirmando que "flexível demais" é
sempre relativo à quantidade de ruído (e de dados) disponível.

## Advanced approach

### 5. Framing a problem: target, moment, metric, baseline

"Reduzir o número de pacientes que faltam às consultas" é um desejo, não um problema de ML. Antes de
tocar nos dados, quatro perguntas:

- **target:** o que exatamente é previsto (e portanto a tarefa) — faltou ou não: classificação.
- **moment:** quando a previsão é feita, e portanto quais atributos são permitidos — 48h antes
  (idade, faltas passadas, dias desde o agendamento), nunca uma informação só conhecida depois
  (vazamento/*leakage*).
- **metric:** como o sucesso é medido, nos termos de quem paga pelos erros.
- **baseline:** o que a resposta mais simples e razoável conseguiria — sem baseline não há como saber
  se 0,80 é impressionante ou pior que nada.

### 6. The goal, written down

$$\text{Meta: minimizar } \mathbb{E}_{(x,y)\sim\mathcal{D}}[L(f(x),y)] \text{ observando apenas uma amostra finita.}$$

A coluna "new" do passo 3 é essa esperança estimada com 1000 sorteios — só foi possível porque
tínhamos a função `draw`. Com dados reais, um conjunto de teste faz esse papel, e só estima a
esperança honestamente se o modelo nunca o viu e se $\mathcal{D}$ não muda depois do deploy (*drift*).

### 7. No free lunch

**Predict (passo 5).** Dois mundos — um reto, outro ondulado (a curva deste notebook) — e três
modelos: a média de treino, uma reta e 5 vizinhos mais próximos. Qual modelo vence em cada mundo?
Algum vence nos dois?

**Resposta (antes de rodar):** Espero que a reta vença no mundo reto (ela assume exatamente esse
formato) e perca no ondulado; e que os vizinhos vençam no ondulado e percam no reto. Não espero que
nenhum modelo vença nos dois — essa é a ideia central do teorema No Free Lunch.

In [8]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

worlds = {'straight': lambda x: 2 * x, 'wiggly': true_curve}

def draw_world(world, n, seed):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    return x.reshape(-1, 1), worlds[world](x) + rng.normal(0, 0.3, n)

models = {
    'mean (baseline)': DummyRegressor(),                    # always answers the training mean
    'straight line': LinearRegression(),
    '5 nearest neighbors': KNeighborsRegressor(n_neighbors=5),   # averages the 5 closest training points
}
print(f"{'error on new points':22s}{'straight':>10s}{'wiggly':>10s}")
for name, model in models.items():
    errors = []
    for world in worlds:
        X, y = draw_world(world, 30, seed=0)
        X_future, y_future = draw_world(world, 1000, seed=1)
        model.fit(X, y)
        errors.append(np.mean((y_future - model.predict(X_future)) ** 2))
    print(f"{name:22s}{errors[0]:10.3f}{errors[1]:10.3f}")

error on new points     straight    wiggly
mean (baseline)            0.464     0.364
straight line              0.101     0.343
5 nearest neighbors        0.131     0.128


**Resposta (depois de rodar):** Confirmado — a reta vence no mundo reto (0,101) e quase só
empata com o baseline no ondulado (0,343 contra 0,364, nenhuma vitória clara); os 5 vizinhos vencem no
ondulado (0,128) e perdem no reto (0,131 contra 0,101). Nenhuma linha é a melhor nas duas colunas:
cada modelo carrega um viés indutivo que combina com um tipo de mundo e não com outro — exatamente o
teorema No Free Lunch (Wolpert, 1996): não há modelo universalmente melhor, por isso o curso ensina um
portfólio de famílias, e cada problema precisa de validação para escolher entre elas.

### 8. Ethics and responsibility — a model learns the past

**Predict (passo 6).** 4000 decisões de empréstimo passadas: pontuações de 0 a 100, distribuídas de
forma igual nos grupos A e B, mas os analistas aprovaram A a partir de 50 e B só a partir de 65. Uma
árvore pequena é treinada para imitá-los. Entre pontuações de 50 a 64, que fração de cada grupo o
modelo vai aprovar?

**Resposta (antes de rodar):** Espero que o modelo reproduza quase exatamente a regra dupla dos
analistas: aprovar quase todo o grupo A e quase nenhum do grupo B nessa faixa de pontuação, porque o
modelo está apenas imitando os rótulos (decisões passadas), não descobrindo nenhuma "verdade" sobre
capacidade de pagamento.

In [9]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(0)
def applicants(n):
    group_B = rng.integers(0, 2, n)                               # 1 = group B, 0 = group A
    return pd.DataFrame({
        'score': rng.integers(0, 101, n),                         # ability to repay, 0 to 100: same in both groups
        'group_B': group_B,
        'zone': (rng.random(n) < np.where(group_B == 1, 0.8, 0.2)).astype(int),  # where they live (simulation)
    })

past = applicants(4000)
# the officers approved group A from a score of 50 up, but group B only from 65 up
past['approved'] = (past['score'] >= np.where(past['group_B'] == 1, 65, 50)).astype(int)

model = DecisionTreeClassifier(max_depth=3, random_state=0)      # a black box for now
model.fit(past[['score', 'group_B']], past['approved'])

new = applicants(2000)
new['approved'] = model.predict(new[['score', 'group_B']])

def approval(df):
    rate = df.groupby('group_B')['approved'].mean()
    return f"group A {rate[0]:.2f} · group B {rate[1]:.2f}"

print('approved, all new applicants: ', approval(new))
print('approved, scores 50 to 64:    ', approval(new[new['score'].between(50, 64)]))

approved, all new applicants:  group A 0.50 · group B 0.34
approved, scores 50 to 64:     group A 1.00 · group B 0.00


**Resposta (depois de rodar):** Confirmado quase exatamente — entre 50 e 64, o modelo aprova
1,00 do grupo A e 0,00 do grupo B. No geral a diferença parece moderada (0,50 contra 0,34), porque a
maioria das pontuações cai onde as duas regras concordam; na faixa de 50–64 a diferença é total. O
modelo é um ótimo aluno — reproduz os rótulos sem erro — e esse é exatamente o problema: os rótulos
eram decisões tendenciosas. Nada no código menciona discriminação; a árvore achou a divisão em
`group_B` porque era a melhor forma de imitar os analistas.

**Simulação: apagar a coluna.** Treinar a mesma árvore só com `score` e `zone` (sem `group_B`) remove
o viés?

**Resposta (antes de rodar):** Não deveria remover — `zone` está fortemente correlacionada com
`group_B` (80% do grupo B mora na zona 1, contra 20% do grupo A), então ela funciona como um *proxy*
para o grupo removido, e o modelo deve continuar discriminando através dela.

In [10]:
blind = DecisionTreeClassifier(max_depth=3, random_state=0)
blind.fit(past[['score', 'zone']], past['approved'])             # no group column
new['approved'] = blind.predict(new[['score', 'zone']])
print('share of group B in zone 1:', round(new.loc[new['zone'] == 1, 'group_B'].mean(), 2))
print('approved, all new applicants: ', approval(new))
print('approved, scores 50 to 64:    ', approval(new[new['score'].between(50, 64)]))

share of group B in zone 1: 0.82
approved, all new applicants:  group A 0.48 · group B 0.37
approved, scores 50 to 64:     group A 0.84 · group B 0.19


**Resposta (depois de rodar):** Confirmado — mesmo sem ver o grupo, o modelo aprova 0,84 do
grupo A e apenas 0,19 do grupo B com as mesmas pontuações (50–64), quase tão discriminatório quanto
antes. 82% da zona 1 é grupo B, então "zona 1" é quase "grupo B": apagar a coluna sensível só tornou o
viés mais difícil de ver; medir as decisões por grupo, como as duas linhas acima fazem, é o que
revela.

### 9. Feedback loops — a model that writes its own data

**Predict (passo 7).** A patrulha da seção 9, por 30 dias em vez de 10. Depois de um mês, qual bairro
os registros dizem ser mais perigoso, e por quanto?

**Resposta (antes de rodar):** Como o modelo sempre manda a patrulha para onde já há mais registros,
e só a patrulha registra incidentes de forma completa, espero que o bairro A (que já começa na
frente, 55 contra 45) acumule uma vantagem registrada cada vez maior, mesmo que B tenha, na verdade,
mais incidentes por dia (12 contra 10) — o laço se realimenta.

In [11]:
true_per_day = {'A': 10, 'B': 12}          # incidents a day: B actually has more
records = {'A': 55, 'B': 45}               # the history the model starts from
for day in range(1, 31):
    hotspot = max(records, key=records.get)          # the "model": where more crime was recorded
    for hood in records:
        if hood == hotspot:
            records[hood] += true_per_day[hood]      # the patrol records every incident it sees
        else:
            records[hood] += 2                       # elsewhere only residents' calls: 2 a day
    if day in (1, 2, 3, 10, 30):
        print(f"day {day:2d}: patrol in {hotspot} · records A {records['A']:3d} · B {records['B']:3d}")

day  1: patrol in A · records A  65 · B  47
day  2: patrol in A · records A  75 · B  49
day  3: patrol in A · records A  85 · B  51
day 10: patrol in A · records A 155 · B  65
day 30: patrol in A · records A 355 · B 105


**Resposta (depois de rodar):** Confirmado — depois de 30 dias os registros dizem que A tem
3,4 vezes o "crime" de B (355 contra 105), e a razão só cresce: a patrulha nunca é enviada a B, então
B nunca acumula registros reais, apesar de ter mais incidentes verdadeiros todos os dias. Cada
registro é um incidente real, mas os registros mentem sobre *onde* está o crime, porque é o próprio
modelo que decide onde olhar. Retreinar não resolve nada, porque cada novo modelo é treinado nos
registros que o modelo anterior moldou.

**Simulação: olhar onde você não olha.** Explorando um dia em cada cinco (indo para o bairro menos
visitado) e, nos demais dias, indo para onde a taxa de incidentes **por dia patrulhado** foi maior —
isso corrige a estimativa?

**Resposta (antes de rodar):** Deveria corrigir: explorar garante que B seja visitado o suficiente
para gerar registros reais, e dividir pelos dias patrulhados (em vez de usar o total bruto) remove a
vantagem artificial que A acumulou só por ter sido visitado mais vezes.

In [12]:
seen = {'A': 0, 'B': 0}                    # incidents the patrol itself recorded
days = {'A': 0, 'B': 0}                    # days the patrol spent there
for day in range(1, 31):
    if day % 5 == 1:                       # every fifth day, look where you did not look
        hotspot = min(days, key=days.get)
    else:                                  # otherwise go where the patrol saw the most per day
        hotspot = max(days, key=lambda h: seen[h] / days[h] if days[h] else 0)
    seen[hotspot] += true_per_day[hotspot]
    days[hotspot] += 1
print('days patrolled:', days)
print('incidents per patrolled day:', {h: seen[h] / days[h] for h in days})

days patrolled: {'A': 9, 'B': 21}
incidents per patrolled day: {'A': 10.0, 'B': 12.0}


**Resposta (depois de rodar):** Confirmado — a patrulha explora A no dia 1 e B no dia 6 (a
primeira visita a B já mostra 12 incidentes contra os 10 de A), e a partir do dia 7 passa a visitar
B com mais frequência (21 dias contra 9). As duas estimativas finais, 10,0 e 12,0 incidentes por dia,
batem exatamente com a verdade simulada (`true_per_day`). As duas mudanças eram necessárias: só
explorar ainda compararia totais brutos (que A sempre venceria); só dividir pelos dias patrulhados,
sem nunca visitar B, nunca geraria dados de B para dividir.

**Para encerrar a aula:**

1. Nem a reta nem os vizinhos vencem nos dois mundos do passo 5 porque cada um carrega um viés
   indutivo diferente (mudança a taxa constante vs. pontos próximos têm respostas próximas), e esses
   viéses combinam com um tipo de mundo e não com o outro.
2. Apagar `group_B` não removeu o viés porque `zone` é um *proxy* fortemente correlacionado com o
   grupo — a árvore aprendeu a mesma regra através da variável substituta.
3. O modelo de patrulha do passo 7 errou ao confundir "onde há mais registros" com "onde há mais
   crime": ele nunca mediu por unidade de esforço (incidentes por dia patrulhado), só o total bruto,
   que cresce simplesmente por patrulhar mais um lugar.

### Minha vez (avançado)

**O futuro fora do passado.** Sorteando os pontos novos do mundo reto fora da faixa de treino (entre
1 e 1,5 em vez de 0 e 1), a reta deve continuar vencendo com folga: ela extrapola corretamente porque
sua suposição ($y=2x$) vale fora do intervalo de treino também. Os vizinhos não conseguem: fora do
intervalo [0,1] não há nenhum ponto de treino próximo, então o modelo de vizinhos simplesmente repete
a resposta do ponto de treino mais próximo da borda — uma constante, não a reta crescente verdadeira.

**Dois tipos de justiça.** Sim, um modelo pode ter taxas de aprovação iguais no total entre os grupos
e ainda tratar pontuações iguais de forma diferente — basta que as diferenças se cancelem em outras
faixas de pontuação (o caso clássico do *paradoxo de Simpson* aplicado a métricas de justiça:
igualdade agregada escondendo desigualdade condicional).

**Patrulhas proporcionais.** Dividindo o tempo de patrulha proporcionalmente aos registros atuais, a
fração de A deve se mover **em direção à verdade** (10/22 ≈ 45,5%) em vez de crescer sem limite ou
ficar presa no ponto de partida: como os dois bairros sempre recebem *algum* tempo de patrulha
(proporcional, nunca zero), ambos acumulam registros reais o suficiente para a proporção não
divergir, embora ainda fique distorcida pelo ponto de partida inicial (55/45) por um tempo, até se
estabilizar perto da proporção real de incidentes.

In [13]:
records_p = {'A': 55.0, 'B': 45.0}
for day in range(1, 31):
    total = records_p['A'] + records_p['B']
    share_A = records_p['A'] / total
    records_p['A'] += true_per_day['A'] * share_A
    records_p['B'] += true_per_day['B'] * (1 - share_A)
    if day in (1, 2, 3, 10, 30):
        print(f"day {day:2d}: share A={share_A:.3f} · records A {records_p['A']:.1f} · B {records_p['B']:.1f}")
print('fração verdadeira de A nos incidentes:', true_per_day['A'] / sum(true_per_day.values()))

day  1: share A=0.550 · records A 60.5 · B 50.4
day  2: share A=0.546 · records A 66.0 · B 55.9
day  3: share A=0.541 · records A 71.4 · B 61.4
day 10: share A=0.520 · records A 108.4 · B 101.0
day 30: share A=0.487 · records A 208.5 · B 220.8
fração verdadeira de A nos incidentes: 0.45454545454545453


**Confirmado:** a fração de A nos registros se move de 55% para perto de 45,5% (a fração
verdadeira de incidentes que são de A, $10/22$), em vez de divergir como no esquema "sempre vai para
quem tem mais registros" — patrulhar proporcionalmente ainda favorece um pouco quem começou na
frente, mas não institui um laço que se autoalimenta sem limite.

## Conclusões

- Reproduzi, colando os blocos de código da página na ordem exata (passos 1 a 7), **todos os números
  publicados** na aula de ML Landscape — da curva com ruído 0,3 ao laço de realimentação da
  patrulha — porque todo o código usa `seed`/`random_state` fixos.
- O experimento do passo 3 mostra numericamente o U do erro em dados novos: piso 0,09 (grau 3–6),
  explodindo para 455 milhões no grau 14 com só 15 pontos, e desaparecendo com 100 pontos — "flexível
  demais" é relativo à quantidade de dados.
- O memorizador do passo 4 prova por que a regra de ouro existe: erro de treino 0,000 tanto com sinal
  real quanto com rótulos embaralhados — o erro de treino de um modelo flexível não discrimina entre
  aprender e memorizar.
- No Free Lunch (passo 5) apareceu em números concretos: a reta vence no mundo reto (0,101) e os
  vizinhos no ondulado (0,128), e nenhum dos dois vence nos dois mundos.
- O experimento do empréstimo (passo 6) mostra que apagar a coluna sensível (`group_B`) não remove o
  viés quando existe um *proxy* (`zone`, 82% correlacionado): a taxa de aprovação do grupo B na faixa
  50–64 continua baixíssima (0,19), quase tão ruim quanto com a coluna original (0,00).
- O laço de realimentação da patrulha (passo 7) mostra que medir o total bruto registrado, quando o
  próprio modelo decide o que é registrado, amplifica um viés inicial pequeno (55/45) até um
  desequilíbrio enorme (355/105) — e que medir por unidade de esforço, com exploração, corrige a
  estimativa de volta para a verdade (10 e 12 incidentes/dia).